In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [3]:
!pip install kaggle

In [8]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:01<00:00, 2.30MB/s]



In [10]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [11]:
import pandas as pd
train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

df = train.merge(stores, on="Store", how="left")

df = df.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

In [12]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=223ba49c-1bec-49e3-8ba4-ff792d7618eb&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1ea7350df7bea011e95102dcf9f7b52e8d91f518643637c27b654424615068f8




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [14]:

import pandas as pd
import numpy as np

from prophet import Prophet

from sklearn.metrics import mean_squared_error, mean_absolute_error

import mlflow
import mlflow.prophet


target = "Weekly_Sales"

df = df.sort_values("Date")

df["IsHoliday"] = df["IsHoliday"].astype(int)


split_1 = df["Date"].quantile(0.8)
split_2 = df["Date"].quantile(0.9)


train_df = df[df["Date"] < split_1]

val_df = df[
    (df["Date"] >= split_1) &
    (df["Date"] < split_2)
]

test_df = df[
    df["Date"] >= split_2
]




X_train = train_df.drop(
    columns=[target, "Date"]
)

y_train = train_df[target]


X_val = val_df.drop(
    columns=[target, "Date"]
)

y_val = val_df[target]


X_test = test_df.drop(
    columns=[target, "Date"]
)

y_test = test_df[target]




regressors = [
    "IsHoliday",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment"
]


def prepare_prophet_data(data, target):

    prophet_df = (
        data[
            ["Date", target] + regressors
        ]
        .groupby("Date")
        .agg({
            target: "sum",
            "IsHoliday": "max",
            "Temperature": "mean",
            "Fuel_Price": "mean",
            "CPI": "mean",
            "Unemployment": "mean"
        })
        .reset_index()
    )


    prophet_df = prophet_df.rename(
        columns={
            "Date": "ds",
            target: "y"
        }
    )

    prophet_df = prophet_df.sort_values("ds")

    return prophet_df



train_prophet = prepare_prophet_data(
    train_df,
    target
)

val_prophet = prepare_prophet_data(
    val_df,
    target
)

test_prophet = prepare_prophet_data(
    test_df,
    target
)


print(train_prophet.head())



mlflow.set_experiment(
    "Prophet_Training"
)


with mlflow.start_run(
    run_name="Prophet_Additive_vs_Multiplicative"
):



    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=0.01,
        seasonality_prior_scale=20
    )



    for reg in regressors:
        model.add_regressor(reg)



    model.fit(
        train_prophet
    )



    train_forecast = model.predict(
        train_prophet[["ds"] + regressors]
    )

    y_train_pred = train_forecast["yhat"].values
    y_train_true = train_prophet["y"].values


    train_rmse = np.sqrt(
        mean_squared_error(
            y_train_true,
            y_train_pred
        )
    )

    train_mae = mean_absolute_error(
        y_train_true,
        y_train_pred
    )


    print(f"TRAIN RMSE: {train_rmse:.4f}")
    print(f"TRAIN MAE: {train_mae:.4f}")


    val_forecast = model.predict(
        val_prophet[["ds"] + regressors]
    )

    y_val_pred = val_forecast["yhat"].values
    y_val_true = val_prophet["y"].values


    val_rmse = np.sqrt(
        mean_squared_error(
            y_val_true,
            y_val_pred
        )
    )


    val_mae = mean_absolute_error(
        y_val_true,
        y_val_pred
    )


    print(f"VALIDATION RMSE: {val_rmse:.4f}")
    print(f"VALIDATION MAE: {val_mae:.4f}")




    mlflow.log_params(
        {
            "model": "Prophet",
            "yearly_seasonality": True,
            "weekly_seasonality": True,
            "daily_seasonality": False,
            "seasonality_mode": "additive",

            "regressors": ",".join(regressors)
        }
    )




    mlflow.log_metrics(
        {
            "train_rmse": train_rmse,
            "train_mae": train_mae,

            "validation_rmse": val_rmse,
            "validation_mae": val_mae
        }
    )
    mlflow.prophet.log_model(
        model,
        "prophet_model"
    )

          ds            y  IsHoliday  Temperature  Fuel_Price         CPI  \
0 2010-02-05  49750740.50          0    33.277942    2.717869  167.398405   
1 2010-02-12  48336677.63          1    33.361810    2.696102  167.384138   
2 2010-02-19  48276993.78          0    37.038310    2.673666  167.338966   
3 2010-02-26  43968571.13          0    38.629563    2.685642  167.691019   
4 2010-03-05  46871470.30          0    42.373998    2.731816  167.727351   

   Unemployment  
0      8.576731  
1      8.567309  
2      8.576351  
3      8.561375  
4      8.572689  
TRAIN RMSE: 3845744.0451
TRAIN MAE: 2269946.6021
VALIDATION RMSE: 2226023.5982
VALIDATION MAE: 1948249.0605


2026/07/12 09:53:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Prophet_Additive_vs_Multiplicative at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6/runs/fa705953007b4edfb85be5b242c4b8cf
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6
